# ProtoHedge Asian-Call Real-Data Panel

This notebook runs the full 10-ticker real-data ProtoHedge panel using an **Asian short-call liability** on the same historical episode set used in the European-call study. The hedge instruments remain spot and the observed vanilla ATM call from the panel data.

The notebook is structured to:
1. run the full per-ticker ProtoHedge sweep,
2. aggregate cross-ticker results,
3. generate paper-ready tables and figures,
4. verify prototype interpretability on a representative ticker.


In [ ]:

from pathlib import Path
import sys
import os
import shutil
import json

cwd = Path.cwd().resolve()
if (cwd / 'world_real_torch.py').exists():
    REPO_ROOT = cwd
elif (cwd.parent / 'world_real_torch.py').exists():
    REPO_ROOT = cwd.parent
else:
    raise RuntimeError(f'Could not locate repo root from {cwd}')

os.environ.setdefault('MPLCONFIGDIR', str((REPO_ROOT / '.matplotlib-cache').resolve()))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import display, Markdown

REPO_PARENT = REPO_ROOT.parent
repo_parent_str = str(REPO_PARENT)
sys.path = [p for p in sys.path if Path(p or '.').resolve() != REPO_PARENT]
sys.path.insert(0, repo_parent_str)
for mod_name in list(sys.modules):
    if mod_name == 'deephedging' or mod_name.startswith('deephedging.'):
        del sys.modules[mod_name]

from deephedging.real_data_sweep_torch import run_real_data_sweep, MODEL_FEATURES
from deephedging.real_data_analysis_torch import (
    evaluate_saved_artifact,
    evaluate_saved_baselines,
    list_saved_artifacts,
    build_regime_frame,
    summarize_by_regime,
    prototype_usage_table,
    prototype_usage_by_regime,
)
from deephedging.prototype_extraction_torch import extract_agent_feature_matrix
from deephedging.proto_analysis_torch import load_prototype_payload

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 220)
plt.rcParams['figure.dpi'] = 120

FINAL_RUN_ROOT = REPO_ROOT / '.deephedging_real_runs' / 'final_paper_asian_panel_smoke'
FINAL_RUN_ROOT.mkdir(parents=True, exist_ok=True)
PAPER_ASSET_DIR = REPO_ROOT / 'paper' / 'final_notebook_assets'
PAPER_ASSET_DIR.mkdir(parents=True, exist_ok=True)
ASIAN_PANEL_SUBDIR = 'asian_real_data_panel'
ASIAN_PANEL_ASSET_DIR = PAPER_ASSET_DIR / ASIAN_PANEL_SUBDIR
ASIAN_PANEL_ASSET_DIR.mkdir(parents=True, exist_ok=True)


def _unique_path(path: Path) -> Path:
    path = Path(path)
    if not path.exists():
        return path
    stem = path.stem
    suffix = path.suffix
    i = 2
    while True:
        cand = path.with_name(f"{stem}_{i}{suffix}")
        if not cand.exists():
            return cand
        i += 1


def save_current_figure(name, subdir=ASIAN_PANEL_SUBDIR, dpi=180):
    subdir_path = PAPER_ASSET_DIR / subdir
    subdir_path.mkdir(parents=True, exist_ok=True)
    path = _unique_path(subdir_path / f'{name}.png')
    plt.gcf().savefig(path, dpi=dpi, bbox_inches='tight')
    print('saved figure:', path)
    return path


def save_dataframe(df, name, subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False):
    subdir_path = PAPER_ASSET_DIR / subdir
    subdir_path.mkdir(parents=True, exist_ok=True)
    csv_path = subdir_path / f'{name}.csv'
    df.to_csv(csv_path, index=index)
    print('saved table:', csv_path)
    if latex:
        tex_path = subdir_path / f'{name}.tex'
        try:
            tex = df.to_latex(index=index, escape=False, float_format=lambda x: f"{x:.6f}" if isinstance(x, (float, np.floating)) else str(x))
            tex_path.write_text(tex)
            print('saved latex:', tex_path)
        except Exception as exc:
            print('latex export skipped:', exc)
    return csv_path


def maybe_copy(src, subdir=f'{ASIAN_PANEL_SUBDIR}/copied_assets'):
    src = Path(src)
    if not src.exists():
        print('missing asset:', src)
        return None
    dst_dir = PAPER_ASSET_DIR / subdir
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = _unique_path(dst_dir / src.name)
    shutil.copy2(src, dst)
    print('copied asset:', dst)
    return dst


def expanded_feature_cols(feature_names, world, input_dim):
    per_step = world.data.features.per_step
    n_inst = int(world.nInst)
    cols = []
    count = 0
    for name in sorted(feature_names or ['price', 'delta', 'time_left']):
        if name in per_step:
            arr = np.asarray(per_step[name])
            width = 1 if arr.ndim == 2 else int(arr.shape[-1])
        elif name in ['delta', 'action']:
            width = n_inst
        elif name in ['pnl', 'cost']:
            width = 1
        else:
            width = 1
        if width == 1:
            cols.append(name)
            count += 1
        else:
            for j in range(width):
                cols.append(f'{name}_{j}')
                count += 1
    if count != int(input_dim):
        cols = [f'feature_{i}' for i in range(int(input_dim))]
    return cols


def select_best_row(best_df, preferred_selection, fallback_selections=()):
    for selection in (preferred_selection, *fallback_selections):
        sub = best_df[best_df['selection'] == selection]
        if not sub.empty:
            return sub.iloc[0]
    raise ValueError(f'Could not find preferred selection {preferred_selection} in best_df')

print('repo root:', REPO_ROOT)
print('final run root:', FINAL_RUN_ROOT)
print('paper assets:', PAPER_ASSET_DIR)


## Controls

The defaults below are set for the **full Asian-call panel run**.


In [ ]:

RUN_PANEL_SWEEP = True


## Panel Setup

This section loads the processed 10-ticker panel and configures the Asian-call sweep.


In [ ]:

PANEL_DIR = REPO_ROOT / 'Data' / 'NEW_PANEL'
MANIFEST_PATH = PANEL_DIR / 'panel_manifest.csv'
assert MANIFEST_PATH.exists(), f'Missing panel manifest: {MANIFEST_PATH}'
manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest.sort_values('ticker').reset_index(drop=True)
display(manifest)

PANEL_OUTPUT_ROOT = FINAL_RUN_ROOT / 'panel_sweeps'
PANEL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PANEL_SUMMARY_DIR = PANEL_OUTPUT_ROOT / 'panel_summary'
PANEL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

PANEL_TICKERS = ['SPY']   # smoke test
SEEDS = (1234,)
EPOCHS = 1
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
NORMALIZE_REAL_WORLD = True
HEDGE_MODE = 'step'
USE_POSITION_BOUNDS = True
SKIP_COMPLETED_TICKERS = False
FORCE_RERUN_TICKERS = set()
PANEL_COMPLETION_FILES = (
    'sweep_metrics.csv',
    'sweep_test_summary.csv',
    'paper_model_comparison.csv',
    'paper_best_models.csv',
)

LIABILITY_TYPE = 'asian_call'
ASIAN_AVERAGE_TYPE = 'arithmetic'
ASIAN_START_STEP = 0
ASIAN_END_STEP = None

TRADE_BOUNDS = {
    'lbnd_as': -1.0,
    'ubnd_as': 1.0,
    'lbnd_av': -1.0,
    'ubnd_av': 1.0,
}
POSITION_BOUNDS = {
    'lbnd_delta_s': -1.0,
    'ubnd_delta_s': 1.0,
    'lbnd_delta_v': -1.0,
    'ubnd_delta_v': 1.0,
}
TRAIN_SELECTION_CFG = {
    'selection_metric': 'val_loss',
    'selection_alpha_action_abs': 0.005,
    'selection_alpha_delta_abs': 0.010,
    'selection_alpha_bound_occupancy': 0.100,
    'selection_alpha_path_bound_touch': 0.0,
}
TRAIN_REG_CFG = {
    'action_penalty_weight': 0.001,
    'delta_penalty_weight': 0.002,
}
ROBUST_SCREEN_CFG = {
    'max_bound_occupancy': 0.50,
    'max_path_touch_rate': None,
}

PANEL_SWEEP_MODE = 'frontier_only'  # smoke test
if PANEL_SWEEP_MODE == 'full_paper_grid':
    PROTOTYPE_COUNTS = (10, 25, 50, 100)
    PROTOTYPE_SOURCES = ('spot_delta', 'vanilla')
    WEIGHTED_SIMILARITY_OPTIONS = (False, True)
    LEARN_DISTANCE_FEATURE_WEIGHTS_OPTIONS = (False,)
else:
    PROTOTYPE_COUNTS = (10, 25)
    PROTOTYPE_SOURCES = ('spot_delta',)
    WEIGHTED_SIMILARITY_OPTIONS = (False,)
    LEARN_DISTANCE_FEATURE_WEIGHTS_OPTIONS = (False,)

PANEL_BASELINE_LABELS = {
    'unhedged': 'Unhedged',
    'spot_delta': 'Spot-Delta',
    'spot_delta_band': 'Spot-Delta Band',
    'vanilla': 'Vanilla DH',
}
PANEL_PROTO_SELECTION_LABELS = {
    'best_screened_proto_mean': 'ProtoHedge (Mean frontier)',
    'best_screened_proto_cvar05': 'ProtoHedge (Tail frontier)',
    'best_screened_proto_shortfall': 'ProtoHedge (Shortfall frontier)',
}
PANEL_PAPER_LABEL_ORDER = [
    'Unhedged',
    'Spot-Delta',
    'Spot-Delta Band',
    'Vanilla DH',
    'ProtoHedge (Mean frontier)',
    'ProtoHedge (Tail frontier)',
    'ProtoHedge (Shortfall frontier)',
]
DEEP_DIVE_TICKER = 'QQQ'
DEEP_DIVE_SEED = int(SEEDS[0])
DEEP_DIVE_SELECTION = 'best_screened_proto_cvar05'

if PANEL_TICKERS is not None:
    manifest = manifest[manifest['ticker'].isin(PANEL_TICKERS)].copy().reset_index(drop=True)

print('panel tickers:', manifest['ticker'].tolist())
print('panel sweep mode:', PANEL_SWEEP_MODE)
print('liability type:', LIABILITY_TYPE)
print('asian average type:', ASIAN_AVERAGE_TYPE)
print('skip completed tickers:', SKIP_COMPLETED_TICKERS)
print('panel output root:', PANEL_OUTPUT_ROOT)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
show = manifest[['ticker', 'n_episodes', 'n_feature_rows']].copy()
axes[0].bar(show['ticker'], show['n_episodes'])
axes[0].set_title('Episodes per ticker')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar(show['ticker'], show['n_feature_rows'])
axes[1].set_title('Feature rows per ticker')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
save_current_figure('asian_panel_manifest_diagnostics', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


## Run Full Asian Panel Sweep

This is the expensive cell. It runs the full per-ticker ProtoHedge grid with the Asian-call liability. Completed ticker sweeps are skipped automatically on reruns.


In [ ]:

def ticker_output_dir(ticker):
    return PANEL_OUTPUT_ROOT / f'{ticker}_{PANEL_SWEEP_MODE}_{EPOCHS}_{LIABILITY_TYPE}_step_robust'


def ticker_is_complete(ticker):
    out_dir = ticker_output_dir(ticker)
    return all((out_dir / name).exists() for name in PANEL_COMPLETION_FILES)


def load_existing_ticker_metrics(ticker):
    out_dir = ticker_output_dir(ticker)
    metrics_path = out_dir / 'sweep_metrics.csv'
    if metrics_path.exists():
        return pd.read_csv(metrics_path)
    return pd.DataFrame()


def run_one_ticker(row):
    ticker = row['ticker']
    out_dir = ticker_output_dir(ticker)
    out_dir.mkdir(parents=True, exist_ok=True)
    if (
        SKIP_COMPLETED_TICKERS
        and ticker not in FORCE_RERUN_TICKERS
        and ticker_is_complete(ticker)
    ):
        print(f'===== {ticker}: found completed sweep, skipping retrain =====')
        return load_existing_ticker_metrics(ticker)

    print(f'===== {ticker}: running {PANEL_SWEEP_MODE} sweep for {LIABILITY_TYPE} =====')
    metrics_df = run_real_data_sweep(
        data_path=row['episode_path'],
        output_dir=out_dir,
        samples=None,
        train_frac=TRAIN_FRAC,
        val_frac=VAL_FRAC,
        seeds=SEEDS,
        epochs=EPOCHS,
        prototype_counts=PROTOTYPE_COUNTS,
        prototype_sources=PROTOTYPE_SOURCES,
        weighted_similarity_options=WEIGHTED_SIMILARITY_OPTIONS,
        learn_distance_feature_weights_options=LEARN_DISTANCE_FEATURE_WEIGHTS_OPTIONS,
        risk_measures=('cvar',),
        normalize=NORMALIZE_REAL_WORLD,
        hedge_mode=HEDGE_MODE,
        position_bounds=USE_POSITION_BOUNDS,
        trade_bounds=TRADE_BOUNDS,
        cumulative_bounds=POSITION_BOUNDS,
        liability_type=LIABILITY_TYPE,
        asian_average_type=ASIAN_AVERAGE_TYPE,
        asian_start_step=ASIAN_START_STEP,
        asian_end_step=ASIAN_END_STEP,
        selection_metric=TRAIN_SELECTION_CFG['selection_metric'],
        selection_alpha_action_abs=TRAIN_SELECTION_CFG['selection_alpha_action_abs'],
        selection_alpha_delta_abs=TRAIN_SELECTION_CFG['selection_alpha_delta_abs'],
        selection_alpha_bound_occupancy=TRAIN_SELECTION_CFG['selection_alpha_bound_occupancy'],
        selection_alpha_path_bound_touch=TRAIN_SELECTION_CFG['selection_alpha_path_bound_touch'],
        action_penalty_weight=TRAIN_REG_CFG['action_penalty_weight'],
        delta_penalty_weight=TRAIN_REG_CFG['delta_penalty_weight'],
        max_bound_occupancy=ROBUST_SCREEN_CFG['max_bound_occupancy'],
        max_path_touch_rate=ROBUST_SCREEN_CFG['max_path_touch_rate'],
        tuned_baseline_metric='gains_mean',
    )
    return metrics_df

if RUN_PANEL_SWEEP:
    panel_metrics = {}
    for _, row in manifest.iterrows():
        panel_metrics[row['ticker']] = run_one_ticker(row)
else:
    print('RUN_PANEL_SWEEP=False, skipping sweep execution.')


## Aggregate Cross-Ticker Outputs

These cells collect the per-ticker sweep outputs and build the panel summary tables and figures.


In [ ]:

def load_panel_outputs_for_ticker(ticker):
    out_dir = ticker_output_dir(ticker)
    comp_path = out_dir / 'paper_model_comparison.csv'
    best_path = out_dir / 'paper_best_models.csv'
    assert comp_path.exists(), f'Missing paper_model_comparison.csv for {ticker}: {comp_path}'
    assert best_path.exists(), f'Missing paper_best_models.csv for {ticker}: {best_path}'
    comp = pd.read_csv(comp_path)
    best = pd.read_csv(best_path)
    artifact_index = list_saved_artifacts(out_dir)
    return out_dir, comp, best, artifact_index

panel_rows = []
panel_all_rows = []
panel_best_rows = []
for _, mrow in manifest.iterrows():
    ticker = mrow['ticker']
    out_dir, comp, best, artifact_index = load_panel_outputs_for_ticker(ticker)
    comp = comp.copy()
    comp['ticker'] = ticker
    comp['output_dir'] = str(out_dir)
    panel_all_rows.append(comp)

    for model, label in PANEL_BASELINE_LABELS.items():
        sub = comp[comp['model'] == model]
        if sub.empty:
            continue
        row = sub.iloc[0].to_dict()
        row['paper_label'] = label
        row['selection'] = model
        panel_rows.append(row)

    for selection, label in PANEL_PROTO_SELECTION_LABELS.items():
        try:
            best_row = select_best_row(
                best,
                selection,
                fallback_selections=(selection.replace('best_screened_', 'best_'),),
            )
        except Exception:
            continue
        row = best_row.to_dict()
        row['ticker'] = ticker
        row['paper_label'] = label
        row['output_dir'] = str(out_dir)
        panel_rows.append(row)
        panel_best_rows.append(row)

panel_df = pd.DataFrame(panel_rows)
panel_all_models_df = pd.concat(panel_all_rows, axis=0, ignore_index=True)
panel_best_df = pd.DataFrame(panel_best_rows)

panel_df['paper_label'] = pd.Categorical(panel_df['paper_label'], PANEL_PAPER_LABEL_ORDER, ordered=True)
panel_df = panel_df.sort_values(['ticker', 'paper_label']).reset_index(drop=True)
panel_df.to_csv(PANEL_SUMMARY_DIR / 'panel_model_metrics.csv', index=False)
panel_all_models_df.to_csv(PANEL_SUMMARY_DIR / 'panel_all_models.csv', index=False)
panel_best_df.to_csv(PANEL_SUMMARY_DIR / 'panel_best_rows.csv', index=False)
print('saved:', PANEL_SUMMARY_DIR / 'panel_model_metrics.csv')
print('saved:', PANEL_SUMMARY_DIR / 'panel_all_models.csv')
display(panel_df[['ticker', 'paper_label', 'gains_mean_avg', 'gains_mean_std', 'gains_cvar05_avg', 'shortfall_prob_avg', 'pct_at_any_position_bound_avg', 'passes_robust_screen']])


In [ ]:

def metric_heatmap(df, metric, title, fmt='.3f', cmap='viridis'):
    pivot = df.pivot(index='ticker', columns='paper_label', values=metric)
    plt.figure(figsize=(12, max(4, 0.45 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap)
    plt.title(title)
    plt.tight_layout()
    return pivot

paper_subset = panel_df[panel_df['paper_label'].isin(PANEL_PAPER_LABEL_ORDER[:6])].copy()
mean_pivot = metric_heatmap(paper_subset, 'gains_mean_avg', 'Asian-call mean gains by ticker and model', fmt='.3f', cmap='viridis')
save_current_figure('asian_panel_mean_heatmap', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

cvar_pivot = metric_heatmap(paper_subset, 'gains_cvar05_avg', 'Asian-call CVaR 5% by ticker and model', fmt='.3f', cmap='magma')
save_current_figure('asian_panel_cvar_heatmap', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

bound_pivot = metric_heatmap(paper_subset, 'pct_at_any_position_bound_avg', 'Asian-call bound occupancy by ticker and model', fmt='.2f', cmap='rocket_r')
save_current_figure('asian_panel_bound_heatmap', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

vanilla = paper_subset[paper_subset['paper_label'] == 'Vanilla DH'][['ticker', 'gains_mean_avg', 'gains_cvar05_avg', 'pct_at_any_position_bound_avg']].rename(
    columns={
        'gains_mean_avg': 'vanilla_mean',
        'gains_cvar05_avg': 'vanilla_cvar',
        'pct_at_any_position_bound_avg': 'vanilla_bound',
    }
)
proto_gap = paper_subset[paper_subset['paper_label'].isin(['ProtoHedge (Mean frontier)', 'ProtoHedge (Tail frontier)'])].merge(vanilla, on='ticker', how='left')
proto_gap['mean_gap_vs_vanilla'] = proto_gap['gains_mean_avg'] - proto_gap['vanilla_mean']
proto_gap['cvar_gap_vs_vanilla'] = proto_gap['gains_cvar05_avg'] - proto_gap['vanilla_cvar']
proto_gap['bound_gap_vs_vanilla'] = proto_gap['pct_at_any_position_bound_avg'] - proto_gap['vanilla_bound']
proto_gap.to_csv(PANEL_SUMMARY_DIR / 'panel_proto_vs_vanilla_gaps.csv', index=False)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, col, title in [
    (axes[0], 'mean_gap_vs_vanilla', 'Asian-call mean gains gap vs vanilla'),
    (axes[1], 'cvar_gap_vs_vanilla', 'Asian-call CVaR 5% gap vs vanilla'),
    (axes[2], 'bound_gap_vs_vanilla', 'Asian-call bound occupancy gap vs vanilla'),
]:
    for label, grp in proto_gap.groupby('paper_label'):
        ax.plot(grp['ticker'], grp[col], marker='o', label=label)
    ax.axhline(0.0, color='black', linewidth=1, linestyle='--')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
axes[0].legend()
plt.xticks(rotation=45)
plt.tight_layout()
save_current_figure('asian_panel_proto_vs_vanilla_gaps', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


In [ ]:

panel_main = paper_subset[['ticker', 'paper_label', 'gains_mean_avg', 'gains_mean_std', 'gains_cvar05_avg', 'shortfall_prob_avg', 'pct_at_any_position_bound_avg', 'passes_robust_screen']].copy()
save_dataframe(panel_main, 'asian_panel_main_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

def avg_table(df):
    rows = []
    for label, grp in df.groupby('paper_label', observed=True):
        rows.append({
            'paper_label': label,
            'mean_gains_avg': grp['gains_mean_avg'].mean(),
            'mean_gains_std_across_tickers': grp['gains_mean_avg'].std(ddof=0),
            'cvar05_avg': grp['gains_cvar05_avg'].mean(),
            'shortfall_avg': grp['shortfall_prob_avg'].mean(),
            'bound_occupancy_avg': grp['pct_at_any_position_bound_avg'].mean(),
            'n_tickers': grp['ticker'].nunique(),
        })
    out = pd.DataFrame(rows)
    out['paper_label'] = pd.Categorical(out['paper_label'], PANEL_PAPER_LABEL_ORDER, ordered=True)
    return out.sort_values('paper_label').reset_index(drop=True)

panel_average_table = avg_table(paper_subset)
save_dataframe(panel_average_table, 'asian_panel_average_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

def rank_by_ticker(df, metric, ascending):
    ranked = []
    for ticker, grp in df.groupby('ticker'):
        sub = grp[['ticker', 'paper_label', metric]].copy()
        sub['rank'] = sub[metric].rank(ascending=ascending, method='min')
        sub['metric'] = metric
        ranked.append(sub)
    return pd.concat(ranked, ignore_index=True)

rank_mean = rank_by_ticker(paper_subset, 'gains_mean_avg', ascending=False)
rank_cvar = rank_by_ticker(paper_subset, 'gains_cvar05_avg', ascending=False)
rank_bound = rank_by_ticker(paper_subset, 'pct_at_any_position_bound_avg', ascending=True)
rank_df = pd.concat([rank_mean, rank_cvar, rank_bound], ignore_index=True)
rank_summary = rank_df.groupby(['paper_label', 'metric'], observed=True)['rank'].mean().reset_index()
save_dataframe(rank_summary, 'asian_panel_average_rank_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

winner_rows = []
for ticker, grp in paper_subset.groupby('ticker'):
    winner_rows.append({
        'ticker': ticker,
        'best_mean_model': grp.sort_values('gains_mean_avg', ascending=False).iloc[0]['paper_label'],
        'best_cvar_model': grp.sort_values('gains_cvar05_avg', ascending=False).iloc[0]['paper_label'],
        'lowest_bound_model': grp.sort_values('pct_at_any_position_bound_avg', ascending=True).iloc[0]['paper_label'],
    })
winner_table = pd.DataFrame(winner_rows).sort_values('ticker').reset_index(drop=True)
save_dataframe(winner_table, 'asian_panel_winner_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

summary_rows = []
van = paper_subset[paper_subset['paper_label'] == 'Vanilla DH'].set_index('ticker')
for proto_label in ['ProtoHedge (Mean frontier)', 'ProtoHedge (Tail frontier)']:
    sub = paper_subset[paper_subset['paper_label'] == proto_label].set_index('ticker')
    common = sub.index.intersection(van.index)
    tmp = sub.loc[common].copy()
    tmp['vanilla_mean'] = van.loc[common, 'gains_mean_avg']
    tmp['vanilla_cvar'] = van.loc[common, 'gains_cvar05_avg']
    tmp['vanilla_bound'] = van.loc[common, 'pct_at_any_position_bound_avg']
    summary_rows.append({
        'proto_label': proto_label,
        'n_tickers': len(common),
        'wins_mean': int((tmp['gains_mean_avg'] > tmp['vanilla_mean']).sum()),
        'wins_cvar': int((tmp['gains_cvar05_avg'] > tmp['vanilla_cvar']).sum()),
        'lower_bound_occupancy': int((tmp['pct_at_any_position_bound_avg'] < tmp['vanilla_bound']).sum()),
        'avg_mean_gap_vs_vanilla': float((tmp['gains_mean_avg'] - tmp['vanilla_mean']).mean()),
        'avg_cvar_gap_vs_vanilla': float((tmp['gains_cvar05_avg'] - tmp['vanilla_cvar']).mean()),
        'avg_bound_gap_vs_vanilla': float((tmp['pct_at_any_position_bound_avg'] - tmp['vanilla_bound']).mean()),
    })
proto_vs_vanilla_summary = pd.DataFrame(summary_rows)
save_dataframe(proto_vs_vanilla_summary, 'asian_panel_proto_vs_vanilla_summary', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

display(panel_average_table)
display(rank_summary)
display(winner_table)
display(proto_vs_vanilla_summary)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
proto_points = panel_all_models_df[panel_all_models_df['model_family'] == 'proto'].copy()
for ticker, grp in proto_points.groupby('ticker'):
    axes[0].scatter(grp['gains_mean_avg'], grp['gains_cvar05_avg'], alpha=0.35, label=ticker)
axes[0].set_title('Asian-call ProtoHedge mean vs CVaR frontier across tickers')
axes[0].set_xlabel('mean gains')
axes[0].set_ylabel('CVaR 5% gains')
axes[0].grid(True, alpha=0.3)

for ticker, grp in proto_points.groupby('ticker'):
    axes[1].scatter(grp['pct_at_any_position_bound_avg'], grp['gains_mean_avg'], alpha=0.35, label=ticker)
axes[1].set_title('Asian-call bound occupancy vs mean gains across ProtoHedge configs')
axes[1].set_xlabel('bound occupancy')
axes[1].set_ylabel('mean gains')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
save_current_figure('asian_panel_frontier_and_bound_scatter', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


## Deep Dive: Representative Ticker

This section reproduces the regime and interpretability analysis for a representative Asian-call ticker, using the selected ProtoHedge tail frontier.


In [ ]:

out_dir, comp, best, artifact_index = load_panel_outputs_for_ticker(DEEP_DIVE_TICKER)
best_row = select_best_row(best, DEEP_DIVE_SELECTION, fallback_selections=('best_proto_cvar05', 'best_cvar05'))
TARGET_MODEL = str(best_row['model'])
print('deep dive ticker:', DEEP_DIVE_TICKER)
print('selection:', DEEP_DIVE_SELECTION)
print('target model:', TARGET_MODEL)

artifact_rows = artifact_index[(artifact_index['model_name'] == TARGET_MODEL) & (artifact_index['risk_measure'] == 'cvar')].copy()
if 'seed' in artifact_rows.columns and DEEP_DIVE_SEED in artifact_rows['seed'].tolist():
    artifact_dir = Path(artifact_rows[artifact_rows['seed'] == DEEP_DIVE_SEED].iloc[0]['artifact_dir'])
else:
    artifact_dir = Path(artifact_rows.iloc[0]['artifact_dir'])
print('artifact:', artifact_dir)

bundle, proto_result, proto_metrics = evaluate_saved_artifact(artifact_dir, split='test')
baselines = evaluate_saved_baselines(artifact_dir, split='test')
vanilla_rows = artifact_index[(artifact_index['model_name'] == 'vanilla') & (artifact_index['risk_measure'] == 'cvar')].copy()
vanilla_artifact_dir = Path(vanilla_rows[vanilla_rows['seed'] == DEEP_DIVE_SEED].iloc[0]['artifact_dir']) if DEEP_DIVE_SEED in vanilla_rows['seed'].tolist() else Path(vanilla_rows.iloc[0]['artifact_dir'])
vanilla_bundle, vanilla_result, vanilla_metrics = evaluate_saved_artifact(vanilla_artifact_dir, split='test')

result_dict = {
    'proto': proto_result,
    'vanilla': vanilla_result,
    'spot_delta_band': baselines['spot_delta_band']['result'],
    'unhedged': baselines['unhedged']['result'],
}
regime_frame = build_regime_frame(bundle['eval_world'], result_dict)
regime_summary = pd.concat([
    summarize_by_regime(regime_frame, 'return_regime'),
    summarize_by_regime(regime_frame, 'vol_regime'),
    summarize_by_regime(regime_frame, 'drawdown_regime'),
], axis=0, ignore_index=True)
regime_summary.to_csv(PANEL_SUMMARY_DIR / f'{DEEP_DIVE_TICKER}_regime_summary.csv', index=False)
save_dataframe(regime_summary, f'{DEEP_DIVE_TICKER.lower()}_asian_regime_summary', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=False, index=False)
display(regime_summary.head())


In [ ]:

for regime_type in ['return_regime', 'vol_regime', 'drawdown_regime']:
    sub = regime_summary[regime_summary['regime_type'] == regime_type].copy()
    for metric in ['mean', 'cvar05', 'shortfall_prob']:
        pivot = sub.pivot(index='regime', columns='series', values=metric)
        plt.figure(figsize=(8, 3))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
        plt.title(f'{DEEP_DIVE_TICKER} Asian-call: {metric} by {regime_type}')
        plt.tight_layout()
        save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_{regime_type}_{metric}', subdir=ASIAN_PANEL_SUBDIR)
        plt.show()


In [ ]:

top_proto_df, full_proto_df = prototype_usage_table(bundle, proto_result, top_n=10)
usage_by_regime_df = prototype_usage_by_regime(proto_result, regime_frame)
save_dataframe(top_proto_df, f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_top10', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)
save_dataframe(usage_by_regime_df, f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_usage_by_regime', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=False, index=False)
display(top_proto_df)

plt.figure(figsize=(10, 4))
plot_df = top_proto_df.sort_values('usage_mean', ascending=False).head(10)
plt.bar(plot_df['prototype_index'].astype(str), plot_df['usage_mean'])
plt.title(f'{DEEP_DIVE_TICKER} Asian-call: top prototype usage')
plt.xlabel('prototype index')
plt.ylabel('mean weight')
plt.tight_layout()
save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_top_prototype_usage', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

for regime_type in ['return_regime', 'vol_regime', 'drawdown_regime']:
    sub = usage_by_regime_df[usage_by_regime_df['regime_type'] == regime_type].copy()
    pivot = sub.pivot(index='regime', columns='prototype', values='mean_weight')
    plt.figure(figsize=(10, 3))
    sns.heatmap(pivot, annot=False, cmap='magma')
    plt.title(f'{DEEP_DIVE_TICKER} Asian-call: prototype usage by {regime_type}')
    plt.tight_layout()
    save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_{regime_type}_prototype_usage', subdir=ASIAN_PANEL_SUBDIR)
    plt.show()


In [ ]:

usage_sorted = full_proto_df.sort_values('usage_mean', ascending=False).reset_index(drop=True).copy()
usage_sorted['cum_usage_mean'] = usage_sorted['usage_mean'].cumsum()
usage_sorted['k'] = np.arange(1, len(usage_sorted) + 1)
save_dataframe(usage_sorted[['k', 'prototype_index', 'usage_mean', 'cum_usage_mean']].head(50), f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_concentration_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

plt.figure(figsize=(8, 4))
plt.plot(usage_sorted['k'], usage_sorted['cum_usage_mean'], linewidth=2)
plt.axhline(0.8, color='black', linestyle='--', linewidth=1)
plt.xlabel('Top-k prototypes')
plt.ylabel('Cumulative usage mass')
plt.title(f'{DEEP_DIVE_TICKER} Asian-call: prototype concentration curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_concentration', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


In [ ]:

proto_path = bundle['metadata']['prototype_path']
payload = load_prototype_payload(proto_path)
prototypes_scaled = np.asarray(payload['prototypes'], dtype=np.float32)
scaler = payload['scaler']
feature_names = payload.get('feature_names', MODEL_FEATURES)

x_raw, feature_names_sorted = extract_agent_feature_matrix(
    world=bundle['eval_world'],
    result=proto_result,
    feature_names=feature_names,
)
x_scaled = scaler.transform(x_raw)
feature_cols = expanded_feature_cols(feature_names_sorted, bundle['eval_world'], x_raw.shape[1])

n_paths = int(bundle['eval_world'].data.market.hedges.shape[0])
n_steps = int(bundle['eval_world'].data.market.hedges.shape[1])

rows = []
for proto_idx in top_proto_df['prototype_index'].head(4).astype(int):
    dists = np.linalg.norm(x_scaled - prototypes_scaled[proto_idx], axis=1)
    nearest = np.argsort(dists)[:3]
    for rank, flat_idx in enumerate(nearest, start=1):
        path_idx = int(flat_idx // n_steps)
        step_idx = int(flat_idx % n_steps)
        row = {
            'prototype_index': int(proto_idx),
            'nearest_rank': int(rank),
            'distance_scaled': float(dists[flat_idx]),
            'path_index': path_idx,
            'step_index': step_idx,
        }
        for j, col in enumerate(feature_cols):
            row[col] = float(x_raw[flat_idx, j])
        rows.append(row)
nearest_proto_states_df = pd.DataFrame(rows).sort_values(['prototype_index', 'nearest_rank']).reset_index(drop=True)
save_dataframe(nearest_proto_states_df, f'{DEEP_DIVE_TICKER.lower()}_asian_nearest_prototype_states', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)
display(nearest_proto_states_df)

fig, axes = plt.subplots(min(4, len(nearest_proto_states_df['prototype_index'].unique())), 1, figsize=(10, 3.0 * min(4, len(nearest_proto_states_df['prototype_index'].unique()))), sharex=True)
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])
for ax, proto_idx in zip(axes, nearest_proto_states_df['prototype_index'].drop_duplicates().tolist()[:4]):
    row = nearest_proto_states_df[nearest_proto_states_df['prototype_index'] == proto_idx].iloc[0]
    path_idx = int(row['path_index'])
    step_idx = int(row['step_index'])
    spot = np.asarray(bundle['eval_world'].data.features.per_step['spot'])[path_idx]
    running_avg = np.cumsum(spot) / (np.arange(len(spot)) + 1.0)
    strike = float(np.asarray(bundle['eval_world'].data.features.per_path['strike'])[path_idx, 0])
    ax.plot(np.arange(len(spot)), spot, label=f'spot path {path_idx}')
    ax.plot(np.arange(len(running_avg)), running_avg, label='running average', linestyle='--')
    ax.axhline(strike, color='gray', linestyle=':', linewidth=1, label='strike')
    ax.axvline(step_idx, color='black', linestyle='--', linewidth=1)
    ax.set_ylabel('normalized level')
    ax.legend(fontsize=8)
axes[-1].set_xlabel('step')
plt.suptitle(f'{DEEP_DIVE_TICKER} Asian-call: nearest historical episodes for top prototypes', y=1.02)
plt.tight_layout()
save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_nearest_historical_episodes', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


In [ ]:

for _, row in manifest.iterrows():
    out_dir = ticker_output_dir(row['ticker'])
    maybe_copy(out_dir / 'paper_model_comparison.csv', subdir=f'{ASIAN_PANEL_SUBDIR}/per_ticker_csv')
    maybe_copy(out_dir / 'paper_best_models.csv', subdir=f'{ASIAN_PANEL_SUBDIR}/per_ticker_csv')
    maybe_copy(out_dir / 'sweep_test_gains_mean.png', subdir=f'{ASIAN_PANEL_SUBDIR}/per_ticker_plots')
    maybe_copy(out_dir / 'sweep_test_cvar05.png', subdir=f'{ASIAN_PANEL_SUBDIR}/per_ticker_plots')
    maybe_copy(out_dir / 'sweep_test_shortfall_prob.png', subdir=f'{ASIAN_PANEL_SUBDIR}/per_ticker_plots')
    maybe_copy(out_dir / 'sweep_test_bound_saturation.png', subdir=f'{ASIAN_PANEL_SUBDIR}/per_ticker_plots')

asset_manifest = pd.DataFrame(sorted([str(p.relative_to(PAPER_ASSET_DIR)) for p in PAPER_ASSET_DIR.rglob('*') if p.is_file()]), columns=['asset'])
save_dataframe(asset_manifest, 'asian_final_asset_manifest', subdir='.', latex=False, index=False)
display(asset_manifest.head(100))
